# Step 2-6. (심화) Distillation: 작은 모델을 학습시키기
Step 2-5에서 모은 스승의 답으로 제자 모델을 학습시킵니다. 학습은 아이맥 안에서 5분쯤 걸립니다.

## 핵심 개념
- **MLX**: Apple Silicon 맥의 GPU로 모델을 돌리고 학습시키는 Apple의 도구
- **LoRA(로라)**: 모델 전체를 고치지 않고, 옆에 붙인 작은 숫자 묶음(전체의 1% 미만)만 학습하는 방법. 1일차에 본 그 방법입니다
- **Adapter(어댑터)**: 학습으로 만들어진 그 작은 숫자 묶음 파일. 수 MB입니다

## 순서
1. 학습 전의 제자에게 시험을 보게 한다
2. 학습시킨다
3. 학습 후의 제자에게 같은 시험을 보게 한다
4. 스승과 속도를 비교하고, Step 2-2의 지휘자 자리에 제자를 넣는다

In [ ]:
# 필요한 패키지를 설치합니다. 처음에는 1~2분 걸립니다.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "mlx-lm", "ollama", "strands-agents[ollama]", "rdkit", "requests"], check=True)
print("✅ 패키지 준비 완료. 다음 셀로 넘어가세요.")

## 1. 설정
제자 모델은 Google의 `gemma-3-1b` (숫자 10억 개, 약 0.7GB)입니다. 스승 `gemma4:e4b`(9.5GB)의 10분의 1도 안 됩니다.

In [ ]:
import json, os, time, warnings

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")

STUDENT = "mlx-community/gemma-3-1b-it-4bit"   # 제자의 바탕 모델 (약 0.8GB)
# TODO: 제자를 바꿔 보세요. 처음 쓸 때 저절로 내려받습니다. 이 실습은 위의 1B 모델로만 확인했습니다.
#   더 작게: "mlx-community/gemma-3-270m-it-4bit" (4분의 1 크기. 어디까지 되는지 보기)
#   더 크게: "mlx-community/gemma-3-4b-it-4bit"   (약 3GB. 메모리 16GB 이상. 학습이 몇 배 오래 걸립니다)
ADAPTER = "distill/adapter"
ITERS = 600          # 학습 반복 횟수
LEARNING_RATE = 1e-4 # 한 번에 얼마나 크게 고칠지. 너무 크면 결과가 실행마다 크게 흔들립니다

if not os.path.exists("distill/data/train.jsonl"):
    raise SystemExit("❌ 학습 자료가 없습니다. 2-5_distill_data 노트북을 먼저 끝까지 실행해 주세요.")

meta = json.load(open("distill/meta.json", encoding="utf-8"))
test = [json.loads(l)["messages"] for l in open("distill/data/test.jsonl", encoding="utf-8")]
valid = [json.loads(l)["messages"] for l in open("distill/data/valid.jsonl", encoding="utf-8")]
print(f"✅ 학습 자료 확인. 시험 문제: 본 유형 {len(valid)}개, 처음 보는 유형 {len(test)}개")
print(f"   스승: {meta['teacher']}   제자: {STUDENT}")

## 2. 학습 전 시험
제자가 스승과 **똑같은 전문가 조합**을 고르면 정답으로 칩니다. 모델을 처음 내려받느라 1분쯤 걸릴 수 있습니다.

In [ ]:
import sys
from mlx_lm import load, generate

sys.path.insert(0, os.path.abspath(".."))   # 저장소 맨 위의 lab_ui.py(진행 상황을 보여 주는 도우미)를 불러오기 위해
from lab_ui import Status


def clean(text):
    return text.split("<")[0].strip().lower().replace(" ", "")


def exam(model, tokenizer, items, show=0):
    correct, start = 0, time.time()
    status = Status(f"⏳ 제자가 시험을 봅니다 ({len(items)}문제)")
    for i, messages in enumerate(items):
        prompt = tokenizer.apply_chat_template(messages[:1], add_generation_prompt=True, tokenize=False)
        out = clean(generate(model, tokenizer, prompt=prompt, max_tokens=16, verbose=False))
        ok = out == messages[1]["content"]
        correct += ok
        if i < show:
            q = messages[0]["content"].split("질문: ")[1]
            print(f"   {'⭕' if ok else '❌'} {q[:34]:<36} 제자: {out[:34]:<36} 스승: {messages[1]['content']}")
        status.update(f"⏳ 제자가 시험 보는 중: {i + 1}/{len(items)}문제  ({status.seconds():.0f}초)")
    status.done(f"✅ 시험 끝: {len(items)}문제 가운데 {correct}문제가 스승과 같음  ({status.seconds():.0f}초)")
    return correct / len(items), (time.time() - start) / len(items)


base_model, tokenizer = load(STUDENT)
print("학습 전, 처음 보는 유형의 질문:")
before_test, _ = exam(base_model, tokenizer, test, show=6)
before_valid, _ = exam(base_model, tokenizer, valid)
print(f"\n📊 학습 전 정답률  본 유형 {before_valid:.0%}   처음 보는 유형 {before_test:.0%}")

## 3. 학습
`train_loss`(틀린 정도)가 줄어드는지 보세요. 4\~5분 걸립니다. 그동안 아이맥의 팬이 돌 수 있습니다.

| 설정 | 뜻 |
|---|---|
| `--iters` | 학습 반복 횟수 |
| `--batch-size` | 한 번에 보는 문제 수 |
| `--learning-rate` | 한 번에 얼마나 크게 고칠지 |
| `--num-layers` | 모델의 층 가운데 몇 개에 LoRA를 붙일지 |
| `--mask-prompt` | 질문 부분은 빼고 답 부분만 채점 |

In [ ]:
import subprocess, sys
from agent_team import follow_training, free_gpu, gpu_was_busy     # 진행 상황을 보여 주고, 학습 전에 큰 모델을 메모리에서 내리는 함수

del base_model          # 메모리를 비웁니다
start = time.time()
cmd = [sys.executable, "-m", "mlx_lm", "lora", "--model", STUDENT, "--train", "--data", "distill/data",
       "--iters", str(ITERS), "--batch-size", "8", "--num-layers", "8", "--learning-rate", str(LEARNING_RATE),
       "--adapter-path", ADAPTER, "--steps-per-report", "10", "--steps-per-eval", "200", "--mask-prompt"]
for attempt in (1, 2):
    free_gpu()                          # 큰 모델(스승)을 메모리에서 내려 학습에 자리를 내줍니다
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env={**os.environ, "NO_COLOR": "1", "TERM": "dumb", "PYTHONUNBUFFERED": "1"})
    log = follow_training(proc, ITERS)      # 10번 반복할 때마다 진행 상황이 바뀝니다
    if proc.wait() == 0:
        break
    if attempt == 1 and gpu_was_busy(log):
        print("⚠️  그래픽 장치(GPU)가 바빠서 학습이 멈췄습니다. 웹 화면이나 다른 노트북에서 모델을 쓰고 있다면 잠시 멈춰 주세요. 20초 뒤 한 번 더 해 봅니다.")
        time.sleep(20)
        continue
    raise SystemExit("❌ 학습이 실패했습니다.\n" + "\n".join(log[-15:]))
size_mb = sum(os.path.getsize(os.path.join(ADAPTER, f)) for f in os.listdir(ADAPTER) if f == "adapters.safetensors") / 1e6
print(f"\n✅ 학습 완료  {time.time() - start:.0f}초   Adapter 파일 {size_mb:.1f}MB")

## 4. 학습 후 시험
같은 문제를 다시 풀게 합니다.

In [ ]:
student, tokenizer = load(STUDENT, adapter_path=ADAPTER)
print("학습 후, 처음 보는 유형의 질문:")
after_test, student_sec = exam(student, tokenizer, test, show=6)
after_valid, _ = exam(student, tokenizer, valid)

print(f"\n📊 정답률 (스승과 같은 답을 고른 비율)")
print(f"                  학습 전    학습 후")
print(f"   본 유형          {before_valid:>5.0%}     {after_valid:>5.0%}")
print(f"   처음 보는 유형   {before_test:>5.0%}     {after_test:>5.0%}")

## 5. 스승과 비교: 속도와 크기
같은 질문을 스승과 제자에게 해서 걸린 시간을 봅니다. 아래 질문은 학습 자료와 **말투가 다릅니다.** 이런 질문에서는 제자가 스승과 다르게 고르는 일이 더 자주 생깁니다. 제자는 본 것과 비슷한 범위 안에서만 스승을 따라 합니다.

In [ ]:
import ollama

ROUTER_PROMPT = """너는 신약 개발팀의 지휘자다. 질문에 답하는 데 꼭 필요한 전문가만 고른다.
- chemist: 화합물 구조, 물리화학적 특성, 구조 변경, 유사 화합물
- pharmacologist: 약동학, 약력학, 독성, 부작용, 약물 상호작용, 용량
- regulatory: 임상시험 현황과 설계, 허가 전략, 시장과 경쟁
질문 하나에 1명에서 3명까지 고를 수 있다."""
ROUTER_SCHEMA = {"type": "object", "required": ["experts"], "properties": {"experts": {
    "type": "array", "minItems": 1, "maxItems": 3,
    "items": {"type": "string", "enum": ["chemist", "pharmacologist", "regulatory"]}}}}
ORDER = ["chemist", "pharmacologist", "regulatory"]


def route_with_teacher(question):
    kw = dict(model=meta["teacher"], format=ROUTER_SCHEMA, options={"temperature": 0},
              messages=[{"role": "system", "content": ROUTER_PROMPT}, {"role": "user", "content": question}])
    try:
        r = ollama.chat(think=False, **kw)
    except Exception:
        r = ollama.chat(**kw)
    return sorted(set(json.loads(r["message"]["content"])["experts"]), key=ORDER.index)


def route_with_student(question):
    """증류한 작은 모델이 전문가를 고른다."""
    prompt = tokenizer.apply_chat_template([{"role": "user", "content": meta["instruction"] + question}], add_generation_prompt=True, tokenize=False)
    picked = [e for e in clean(generate(student, tokenizer, prompt=prompt, max_tokens=16, verbose=False)).split(",") if e in ORDER]
    return sorted(set(picked), key=ORDER.index) or ORDER      # 알 수 없는 답이면 전원 호출


# TODO: 질문을 바꿔 보세요!
questions = ["이 화합물을 당뇨 환자에게 써도 될까?", "특허가 끝난 뒤 복제약과 어떻게 경쟁하지?", "물에 잘 안 녹는데 어떻게 고칠까?"]
route_with_teacher(questions[0])      # 스승 모델을 미리 메모리에 올려 둡니다
for q in questions:
    t0 = time.time(); a = route_with_teacher(q); t1 = time.time(); b = route_with_student(q); t2 = time.time()
    print(f"{q}\n   스승 {t1 - t0:.2f}초  {a}\n   제자 {t2 - t1:.2f}초  {b}   {'⭕ 같음' if a == b else '❌ 다름'}\n")

print(f"크기: 스승 약 9,500MB  제자 약 700MB + Adapter {size_mb:.1f}MB")

## 6. 제자를 지휘자 자리에 넣기
Step 2-2의 Multi-Agent를 `drug_agents.py` 파일에 그대로 옮겨 두었습니다. `consult()`의 `router` 자리에 `route_with_student`를 넘기면, **전문가를 고르는 일은 방금 학습시킨 작은 모델이** 하고 분석은 큰 모델이 합니다. 1분쯤 걸립니다.

In [ ]:
import sys

sys.path.insert(0, os.path.abspath(".."))   # 저장소 맨 위의 drug_agents.py를 불러오기 위해
from drug_agents import consult

# TODO: 약 이름(영문)과 질문을 바꿔 보세요!
_ = consult("고령 환자에게 써도 될까?", drug="metformin", router=route_with_student)

## 정리
- 스승의 답 수백 건과 몇 분의 학습으로, 10분의 1 크기 모델이 본 유형의 질문에서는 스승과 대부분 같은 선택을 하게 됐습니다.
- **증류가 잘 되는 일**: 답의 형태가 정해져 있고 반복되는 일(분류, 라우팅, 정보 뽑기)
- **증류로 안 되는 것**: 제자는 스승을 넘지 못합니다. 스승이 틀린 것은 제자도 틀립니다. 처음 보는 유형에서는 정답률이 떨어집니다
- **실무에서는**: 큰 모델로 서비스를 먼저 만들고, 쌓인 기록으로 단순한 부분부터 작은 모델로 바꿉니다. 비용과 속도가 문제일 때 꺼내는 방법입니다

## 더 해 보기
- `ITERS`를 200, 1000으로 바꾸면 정답률이 어떻게 달라지나요?
- `LEARNING_RATE`를 `2e-4`로 올리고 두세 번 다시 학습해 보세요. 결과가 실행마다 크게 달라집니다(발표자 맥에서 15%\~74%)
- 제자를 `gemma-3-270m`(4분의 1 크기)으로 바꾸면 어디까지 되나요?
- Step 2-5에서 질문 틀을 늘리면 "처음 보는 유형"의 정답률이 오르나요?
- GPU 서버(EC2, SageMaker)의 더 큰 모델을 스승으로 쓰려면 `aws/README.md`를 보세요